# Install Necessary Libraries

In [ ]:
%%capture

!pip install torch transformers chromadb datasets qwen-vl-utils hf_xet langchain-chroma langchain-core Pillow llama-cpp-python huggingface-hub python-dotenv

In [ ]:
from huggingface_hub import login
import os

# Colab secrets take priority, then environment variable
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("HF_TOKEN is not set. Add it to Colab secrets or set as an environment variable.")

# (Optional) Install Flash Attention

Makes models run faster with less VRAM usage.

**Requires Ampere or newer GPU — T4 does NOT support Flash Attention.**
Skip this cell if running on T4. Use L4 or better for Flash Attention.

In [4]:
!nvidia-smi

# Uncomment below ONLY if running on Ampere+ GPU (L4, A100, etc.) — NOT T4
# !pip install ninja
# !pip install flash-attn --no-build-isolation

Thu Apr 16 06:54:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   48C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Setup Paths

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from huggingface_hub import snapshot_download

# Detect environment: Colab vs local
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

if IN_COLAB:
    # Clone or pull the repo on Colab
    repo_path = "/content/Mills-Museum"
    if not os.path.exists(repo_path):
        subprocess.run(["git", "clone", "https://github.com/cs2535-oakhoury-2026spring/Mills-Museum.git", repo_path])
    else:
        subprocess.run(["git", "pull"], cwd=repo_path)
else:
    # Running locally — resolve repo root relative to this notebook (colab/ dir)
    repo_path = str(Path(os.getcwd()).parent) if Path(os.getcwd()).name == "colab" else os.getcwd()
    if not os.path.exists(os.path.join(repo_path, "src")):
        repo_path = os.getcwd()

# Download VDB from Hugging Face (cached after first download)
VDB_PATH = snapshot_download(repo_id="KeeganC/mcam-vdb", repo_type="dataset")

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Repo path:   {repo_path}")
print(f"VDB path:    {VDB_PATH}")
print(f"VDB exists:  {os.path.exists(VDB_PATH)}")

# Choose How Many Keywords to Find

In [10]:
TERM_COUNT = 10

# Load Data & Vector Store

Loads the LangChain Chroma vectorstore and AAT term-to-ID mapping. No GPU needed for this cell.

In [8]:
COLLECTION_NAME = "aat_terms"

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from datasets import load_dataset
from datasets.arrow_dataset import Dataset
from huggingface_hub import hf_hub_download
import importlib.util
import torch
import gc
import csv

# Load Qwen embedding wrapper from the official HF repo (cached after first download)
embed_script = hf_hub_download(
    repo_id="Qwen/Qwen3-VL-Embedding-2B",
    filename="qwen3_vl_embedding.py",
)
spec = importlib.util.spec_from_file_location("qwen3_vl_embedding", embed_script)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
Qwen3VLEmbedder = mod.Qwen3VLEmbedder

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load LangChain Chroma vector store (no embedding_function needed for search-by-vector)
vectorstore = Chroma(
    collection_name="aat_terms",
    persist_directory=VDB_PATH,
    collection_metadata={"hnsw:space": "cosine"},
)
print(f"Loaded vectorstore with {vectorstore._collection.count()} documents")

# Load AAT term-to-ID mapping
aat_terms: Dataset = load_dataset(path="KeeganC/aat-museum-subset", split="train")
selection = aat_terms.select_columns(['preferred_term', 'subject_id'])
aat_terms_to_ids: dict[str, str] = {}
for i in selection.iter(1):
    aat_terms_to_ids[i['preferred_term'][0]] = i['subject_id'][0]

# MMR parameters
FETCH_K = TERM_COUNT * 4  # fetch more candidates for MMR diversity selection
LAMBDA_MULT = 0.7         # 1.0 = pure relevance, 0.0 = pure diversity

print("Data loaded. Ready to run pipeline.")

# Load Models & Generate Keywords

Loads the Qwen embedding model and GGUF reranker, then defines the reranking pipeline.

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
import gc
import csv
import base64
import io
import math

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# ── Load embedding model ──
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Qwen embedding model on {device}...")

try:
    import flash_attn
    print("Flash Attention found, proceeding with optimized code path.")
    ATTENTION_IMPLEMENTATION = "flash_attention_2"
except ImportError:
    print("Flash Attention not installed, falling back to default attention implementation.")
    ATTENTION_IMPLEMENTATION = "sdpa"

embedding_model_name = "Qwen/Qwen3-VL-Embedding-2B"

embedding_model = Qwen3VLEmbedder(
    model_name_or_path=embedding_model_name,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    attn_implementation=ATTENTION_IMPLEMENTATION
)

# ── Load reranker as GGUF (Q6_K imatrix quant) ──
print("Downloading Qwen3-VL-Reranker-2B GGUF (Q6_K imatrix)...")

reranker_gguf_path = hf_hub_download(
    repo_id="mradermacher/Qwen3-VL-Reranker-2B-i1-GGUF",
    filename="Qwen3-VL-Reranker-2B.i1-Q6_K.gguf",
)
reranker_mmproj_path = hf_hub_download(
    repo_id="mradermacher/Qwen3-VL-Reranker-2B-GGUF",
    filename="Qwen3-VL-Reranker-2B.mmproj-Q8_0.gguf",
)

print(f"GGUF model: {reranker_gguf_path}")
print(f"mmproj:     {reranker_mmproj_path}")

reranker_llm = Llama(
    model_path=reranker_gguf_path,
    clip_model_path=reranker_mmproj_path,
    n_ctx=4096,
    n_gpu_layers=-1,
    logits_all=False,
    verbose=False,
)

# Find token IDs for "yes" and "no" in the GGUF vocabulary
YES_TOKEN = reranker_llm.tokenize(b"yes", add_bos=False)[0]
NO_TOKEN  = reranker_llm.tokenize(b"no",  add_bos=False)[0]
print(f"Reranker token IDs — yes: {YES_TOKEN}, no: {NO_TOKEN}")


def _pil_to_data_uri(img):
    """Convert a PIL Image to a base64 data URI for llama-cpp vision."""
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f"data:image/png;base64,{b64}"


def rerank_with_gguf(instruction: str, query: dict, documents: list[dict]) -> list[float]:
    """
    Score each document against the query using the GGUF reranker.
    Returns a list of relevance scores (0-1) in the same order as documents.
    """
    img_uri = _pil_to_data_uri(query["image"]) if "image" in query and query["image"] is not None else None
    query_text = query.get("text", "")

    scores = []
    for doc in tqdm(documents, desc="Reranking"):
        doc_text = doc.get("text", "")

        user_content = []
        if img_uri:
            user_content.append({"type": "image_url", "image_url": {"url": img_uri}})
        user_content.append({
            "type": "text",
            "text": (
                f"Instruction: {instruction}\n"
                f"Query: {query_text}\n"
                f"Document: {doc_text}\n"
                "Is the Document relevant to the Query? Answer only 'yes' or 'no'."
            ),
        })

        messages = [
            {"role": "system", "content": "Judge whether the Document is relevant to the Query. Answer only 'yes' or 'no'."},
            {"role": "user", "content": user_content},
        ]

        resp = reranker_llm.create_chat_completion(
            messages=messages,
            max_tokens=1,
            logprobs=True,
            top_logprobs=10,
            temperature=0.0,
        )

        top_lps = resp["choices"][0]["logprobs"]["content"][0]["top_logprobs"]
        lp_map = {entry["token"].strip().lower(): entry["logprob"] for entry in top_lps}

        lp_yes = lp_map.get("yes", -100.0)
        lp_no  = lp_map.get("no",  -100.0)

        score = math.exp(lp_yes) / (math.exp(lp_yes) + math.exp(lp_no))
        scores.append(score)

    return scores


print("Both models loaded and ready!")

# API for Frontend

In [ ]:
# Install the packages needed to run the web server and expose it publicly
!pip install fastapi uvicorn pyngrok nest-asyncio python-multipart langchain-chroma langchain-core

import uvicorn
import nest_asyncio
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok, conf
from PIL import Image
import torch
import io
import os

# nest_asyncio lets uvicorn run inside it without crashing
nest_asyncio.apply()

# Get NGROK_TOKEN from environment variable
conf.get_default().auth_token = os.environ.get('NGROK_TOKEN')

app = FastAPI()

# Without this the browser would block requests from the frontend to this server
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Path to the pre-built React app
DIST_PATH = os.path.join(repo_path, "src", "frontend", "web")

@app.get("/")
async def serve_frontend():
    return FileResponse(f"{DIST_PATH}/index.html")

# Serve the JS/CSS bundles that index.html references
app.mount("/assets", StaticFiles(directory=f"{DIST_PATH}/src/assets"), name="assets")

@app.post("/predict")
async def predict(file: UploadFile = File(...), term_count: int = Form(default=20)):
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")

    art_query = {"image": image, "text": ""}
    query_input = [art_query]

    image_features = embedding_model.process(query_input)
    image_features = torch.nn.functional.normalize(image_features, p=2, dim=1)
    query_embedding = image_features.cpu().float().squeeze().tolist()

    # MMR retrieves term_count keywords while penalizing redundancy
    mmr_docs = vectorstore.max_marginal_relevance_search_by_vector(
        embedding=query_embedding,
        k=term_count,
        fetch_k=term_count * 4,
        lambda_mult=0.96,
    )

    labels = []
    input_docs = []
    for doc in mmr_docs:
        term_label = doc.metadata.get("term_label", doc.page_content.split("\n")[0])
        input_docs.append({"text": doc.page_content})
        labels.append(term_label)

    # Rerank using GGUF model
    scores = rerank_with_gguf(
        instruction="Retrieve Art & Architecture Thesaurus terms relevant to the given image.",
        query=art_query,
        documents=input_docs,
    )

    sorted_results = sorted(zip(scores, labels), reverse=True)

    # Response shape matches what keywordAdapters.js on the frontend expects
    keywords = [
        {"label": label, "score": round(float(score) * 100, 1)}
        for score, label in sorted_results
    ]

    return {"keywords": keywords}

# Static ngrok domain so the URL stays the same every run
public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")
print(f"Open this URL in your browser to use the app!")

# host="0.0.0.0" makes the server reachable from outside the Colab container
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()